# (프로젝트 제목)

**AI활용코딩연습(ZBE135) 최종 프로젝트** · (이름 / 학번)

> 한 줄 요약: (가장 중요한 결론을 숫자와 함께 — 분석을 마친 뒤 작성)

이 노트북은 최종 프로젝트 **템플릿**입니다. 각 절의 안내를 지우고 내용을 채우세요. 예시로 `cafe_sales.csv`를 읽도록 되어 있으니, 자신의 데이터로 바꿉니다.

| 절 | 내용 |
|---|---|
| 0 | 재현성 셀 |
| 1 | 연구 질문 |
| 2 | 데이터 |
| 3 | 정제·파생 변수 |
| 4 | 분석과 테스트 |
| 5 | 시각화 |
| 6 | 결론과 한계 |
| 7 | AI 활용 공개 |

---
## 0. 재현성 셀

결과가 달라졌을 때 **버전·데이터·시드** 중 무엇이 바뀌었는지 확인하는 단서입니다. 지우지 마세요.

In [ ]:
import sys
import hashlib
import platform
from datetime import datetime
from pathlib import Path
import urllib.request

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

try:
    import koreanize_matplotlib
except ModuleNotFoundError:
    %pip install -q koreanize-matplotlib
    import koreanize_matplotlib

SEED = 42
rng = np.random.default_rng(SEED)

# 수업 데이터는 GitHub에서, 내 데이터는 공개 URL이나 노트북과 같은 폴더의 파일로
DATA_URL = "https://raw.githubusercontent.com/MNU-AI-Programming/AI-Assisted-Python-Programming/main/datasets/"

def get_data(name, url=None):
    """같은 폴더(또는 datasets/)에 있으면 그 경로, 없으면 url(기본: 수업 저장소)에서 내려받는다."""
    for folder in [Path("."), Path("datasets"), Path("../../datasets"), Path("../datasets")]:
        if (folder / name).exists():
            return str(folder / name)
    urllib.request.urlretrieve(url or DATA_URL + name, name)
    return name


def fingerprint(path):
    raw = Path(path).read_bytes()
    return {"file": Path(path).name, "bytes": len(raw), "md5": hashlib.md5(raw).hexdigest()[:12]}


print(f"실행 시각: {datetime.now():%Y-%m-%d %H:%M} / Python {sys.version.split()[0]} ({platform.system()})")
print(f"numpy {np.__version__} / pandas {pd.__version__} / matplotlib {matplotlib.__version__} / SEED {SEED}")

---
## 1. 연구 질문

> 데이터로 답할 수 있고, 비교 대상이 있으며, 표·그래프로 보여 줄 수 있는 질문 2~3개

1. (질문 1)
2. (질문 2)

---
## 2. 데이터

| 항목 | 내용 |
|---|---|
| 파일 / 출처 | (예: cafe_sales.csv / 수업용 가상 데이터) |
| 이용 조건 | |
| 한 행의 의미 | (예: 주문 한 건의 메뉴 한 종류) |
| 기간 · 행 수 | |
| 주요 열 | |

In [ ]:
DATA_FILE = get_data("cafe_sales.csv")          # ← 내 데이터로 바꾸세요
print(fingerprint(DATA_FILE))

raw = pd.read_csv(DATA_FILE)
print(raw.shape)
raw.head()

In [ ]:
raw.info()
print(raw.isna().sum()[lambda s: s > 0])

---
## 3. 정제·파생 변수

정제 계획을 표로 먼저 쓰고, 정제는 **함수로**(입력 불변) 작성합니다.

| 열 | 문제 | 처리 | 이유 |
|---|---|---|---|
| | | | |

In [ ]:
def clean(raw):
    """원본 → 분석용 DataFrame (입력은 바꾸지 않는다)."""
    df = raw.copy()
    # 예시: 날짜 변환과 파생 변수 — 내 데이터에 맞게 바꾸세요
    df["date"] = pd.to_datetime(df["date"])
    df["revenue"] = df["unit_price"] * df["qty"]
    return df


df = clean(raw)
print(raw.shape, "→", df.shape)

---
## 4. 분석과 테스트

질문마다 계산 함수를 만들고, **손으로 답을 알 수 있는 작은 데이터**로 테스트합니다.

In [ ]:
def answer_q1(df):
    """질문 1에 답하는 표. (예시: 메뉴별 매출 비중)"""
    out = df.groupby("menu")["revenue"].sum().sort_values(ascending=False).to_frame()
    out["share"] = (out["revenue"] / out["revenue"].sum() * 100).round(1)
    return out


def test_answer_q1():
    tiny = pd.DataFrame({"menu": ["A", "B", "A"], "revenue": [100, 200, 100]})
    t = answer_q1(tiny)
    assert t.loc["A", "share"] == 50.0 and t.loc["B", "share"] == 50.0


test_answer_q1()
print("테스트 통과 ✅")
answer_q1(df).head()

---
## 5. 시각화

제목은 **결론 문장**, 축에는 **단위**, 막대는 **0부터**, 색은 **의미가 있을 때만**.

In [ ]:
q1 = answer_q1(df).head(8).sort_values("share")
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.barh(q1.index, q1["share"], color=["#BBBBBB"] * (len(q1) - 1) + ["#C0392B"])
ax.set_title("(결론 문장으로 바꾸세요)", loc="left", fontweight="bold")
ax.set_xlabel("매출 비중 (%)")
ax.spines[["top", "right"]].set_visible(False)
Path("figures").mkdir(exist_ok=True)
fig.savefig("figures/main.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 6. 결론과 한계

| 결론 | 근거 (표·그래프) | 한계 |
|---|---|---|
| 1. | | |
| 2. | | |

- 상관과 인과를 구분했는가?
- 표본 크기(일수, 인원)를 밝혔는가?

---
## 7. AI 활용 공개

| 단계 | 사용한 도구와 프롬프트 요약 | 채택 | 고친 것과 이유 | 검증 방법 |
|---|---|---|---|---|
| | | | | |

자세한 기록은 `ai_use_log.md`에 남깁니다.

> 제출 전: **런타임 → 다시 시작 및 모두 실행**으로 처음부터 끝까지 오류 없이 실행되는지 확인하세요.